# PPE Detector V2 - DetectionHead A3: Sampling, Crop, and Controlled Multi-Scale
 
 This notebook evaluates the complete small-object training recipe. It combines A1 sampling, A2 object-centered crops, and controlled multi-scale training. These changes affect training only; the deployed DetectionHead architecture remains unchanged.
 
- **A1:** small-object-aware sampling.
- **A2:** object-centered crops with worker context.
- **A3:** A1 and A2 plus controlled multi-scale training.


## 1. Research question and decision rule

**Question:** Can scale-aware data sampling and cropping improve small PPE detection without reducing overall detection quality or increasing edge latency?

The candidate is kept when validation `AP_small` improves by at least 2 percentage points, overall mAP50-95 decreases by no more than 0.5 percentage points, and the model architecture remains unchanged. The current DetectionHead result is stored as the reference row.

In [ ]:
import importlib.util
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

from packaging.version import Version

requirements = []
try:
    if Version(version("ultralytics")) < Version("8.4.190"):
        requirements.append("ultralytics>=8.4.190")
except PackageNotFoundError:
    requirements.append("ultralytics>=8.4.190")
if importlib.util.find_spec("torchmetrics") is None:
    requirements.append("torchmetrics>=1.4")
if importlib.util.find_spec("pycocotools") is None:
    requirements.append("pycocotools>=2.0.7")
if requirements:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *requirements])
print("Dependencies are ready.")

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import platform
import random
import shutil
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import yaml
from PIL import Image
from IPython.display import display
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from ultralytics import YOLO
import ultralytics

EXPERIMENT = "A3_sampling_crop_multiscale"
VALID_EXPERIMENTS = {
    "A1_sampling",
    "A2_sampling_crop",
    "A3_sampling_crop_multiscale",
}
if EXPERIMENT not in VALID_EXPERIMENTS:
    raise ValueError(f"Choose one of {sorted(VALID_EXPERIMENTS)}")

DATASET_ROOT_OVERRIDE = None
OUTPUT_ROOT_OVERRIDE = None
RESUME_WEIGHTS = None  # Set only when resuming the same A1/A2/A3 experiment.
RUN_TRAINING = True
EPOCHS = 100
IMAGE_SIZE = 640
BATCH = 32
SEED = 42
SAVE_PERIOD = 5
MAX_EXTRA_SAMPLES = 4000
MAX_CROPS = 2500
MIN_CROP_SIDE = 256
CROP_CONTEXT = 8.0
MIN_BOX_VISIBILITY = 0.50
TINY_AREA = 16 ** 2
SMALL_AREA = 32 ** 2
TARGET_CLASS_IDS = {0, 1}
CLASS_NAMES = {0: "helmet", 1: "no-helmet", 2: "vest", 3: "no-vest"}
COMPLETE_SOURCES = {"tucje", "gsnvb", "mendeley", "css"}
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

GPU_COUNT = torch.cuda.device_count()
TRAIN_DEVICE = [0, 1] if GPU_COUNT >= 2 else (0 if GPU_COUNT == 1 else "cpu")
EVAL_DEVICE = 0 if GPU_COUNT >= 1 else "cpu"
BATCH = BATCH if GPU_COUNT >= 2 else min(BATCH, 16)
WORKERS = 2 if GPU_COUNT >= 2 else 4
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Ultralytics:", ultralytics.__version__)
print("Experiment:", EXPERIMENT)
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(GPU_COUNT)])
print("Training device:", TRAIN_DEVICE, "| Global batch:", BATCH)

## 2. Locate Dataset V2 and create the experiment workspace

The Kaggle dataset path is detected automatically. All derived crops, manifests, checkpoints, metrics, and figures are written under `/kaggle/working` so the mounted dataset remains unchanged.

In [ ]:
def is_dataset_root(path):
    path = Path(path)
    val_folder = "valid" if (path / "images" / "valid").exists() else "val"
    return (path / "images" / "train").exists() and (path / "images" / val_folder).exists() and (path / "data.yaml").exists()

candidates = []
if DATASET_ROOT_OVERRIDE:
    candidates.append(Path(DATASET_ROOT_OVERRIDE))
candidates.extend([
    Path("/kaggle/input/datasets/linhlaz225/helmet-nohelmet-vest-novest/Dataset_v2"),
    Path("/kaggle/input/helmet-nohelmet-vest-novest/Dataset_v2"),
    Path(r"D:/UTE/TLCN/Dataset/Dataset_v2"),
])
DATASET_ROOT = next((path for path in candidates if is_dataset_root(path)), None)
if DATASET_ROOT is None and Path("/kaggle/input").exists():
    DATASET_ROOT = next((path for path in Path("/kaggle/input").rglob("Dataset_v2") if is_dataset_root(path)), None)
if DATASET_ROOT is None:
    raise FileNotFoundError("Dataset_v2 was not found. Set DATASET_ROOT_OVERRIDE.")

VAL_FOLDER = "valid" if (DATASET_ROOT / "images" / "valid").exists() else "val"
OUTPUT_ROOT = Path(OUTPUT_ROOT_OVERRIDE) if OUTPUT_ROOT_OVERRIDE else Path("/kaggle/working/ppe_detectionhead_small_object_sampling_study")
CONFIG_DIR = OUTPUT_ROOT / "config"
DERIVED_ROOT = OUTPUT_ROOT / "derived"
RUNS_DIR = OUTPUT_ROOT / "runs"
RECOVERY_DIR = OUTPUT_ROOT / "recovery" / EXPERIMENT
RESULTS_DIR = OUTPUT_ROOT / "results" / EXPERIMENT
EXPORT_DIR = OUTPUT_ROOT / "exports"
for folder in (CONFIG_DIR, DERIVED_ROOT, RUNS_DIR, RECOVERY_DIR, RESULTS_DIR, EXPORT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

print("Dataset:", DATASET_ROOT)
print("Validation folder:", VAL_FOLDER)
print("Output:", OUTPUT_ROOT)

## 3. Audit object sizes

Bounding boxes are projected to a 640-pixel training canvas. A target is `tiny` below 16 squared pixels and `small` below 32 squared pixels. Only helmet and no-helmet from complete four-label sources are oversampled; partial-label sources remain available once but are not amplified.

In [ ]:
def source_name(image_path):
    return Path(image_path).stem.split("__", 1)[0].lower()

def label_path_for(image_path):
    image_path = Path(image_path)
    parts = list(image_path.parts)
    image_index = max(index for index, value in enumerate(parts) if value == "images")
    parts[image_index] = "labels"
    return Path(*parts).with_suffix(".txt")

image_rows, box_rows = [], []
for split, folder in {"train": "train", "val": VAL_FOLDER}.items():
    image_dir = DATASET_ROOT / "images" / folder
    for image_path in sorted(path for path in image_dir.iterdir() if path.suffix.lower() in IMAGE_EXTENSIONS):
        label_path = label_path_for(image_path)
        if not label_path.exists():
            raise FileNotFoundError(f"Missing label: {label_path}")
        source = source_name(image_path)
        image_rows.append({
            "split": split, "source": source,
            "image_path": str(image_path.resolve()), "label_path": str(label_path.resolve()),
        })
        for line_number, line in enumerate(label_path.read_text(encoding="utf-8", errors="ignore").splitlines(), 1):
            values = line.split()
            if len(values) != 5:
                raise ValueError(f"Invalid label at {label_path}:{line_number}")
            class_id = int(float(values[0]))
            x, y, width, height = map(float, values[1:])
            area = width * height * IMAGE_SIZE * IMAGE_SIZE
            size_group = "tiny" if area < TINY_AREA else "small" if area < SMALL_AREA else "medium_or_large"
            box_rows.append({
                "split": split, "source": source, "image_path": str(image_path.resolve()),
                "label_path": str(label_path.resolve()), "class_id": class_id,
                "class_name": CLASS_NAMES[class_id], "x": x, "y": y, "w": width, "h": height,
                "area_at_640": area, "size_group": size_group,
            })

images_df = pd.DataFrame(image_rows)
boxes_df = pd.DataFrame(box_rows)
size_table = boxes_df.groupby(["class_name", "size_group"]).size().unstack(fill_value=0).reindex(CLASS_NAMES.values(), fill_value=0)
size_table["tiny_or_small_ratio"] = (size_table.get("tiny", 0) + size_table.get("small", 0)) / size_table.sum(axis=1).clip(lower=1)
display(size_table)
size_table.to_csv(RESULTS_DIR / "object_size_distribution.csv")

plt.figure(figsize=(9, 4))
sns.ecdfplot(data=boxes_df[boxes_df["split"] == "train"], x="area_at_640", hue="class_name", log_scale=True)
plt.axvline(TINY_AREA, color="black", linestyle=":", linewidth=1)
plt.axvline(SMALL_AREA, color="black", linestyle="--", linewidth=1)
plt.title("Object area at a 640-pixel training scale")
plt.tight_layout()
plt.show()

## 4. Build sampling entries and object-centered crops

Every original training image remains available. Up to 4,000 additional sampling entries and 2,500 deterministic crops are added from complete four-label sources. Training then enables controlled multi-scale variation.


In [ ]:
rng = random.Random(SEED)
train_images = images_df[images_df["split"] == "train"].copy()
small_targets = boxes_df[
    (boxes_df["split"] == "train")
    & boxes_df["class_id"].isin(TARGET_CLASS_IDS)
    & boxes_df["size_group"].isin(["tiny", "small"])
    & boxes_df["source"].isin(COMPLETE_SOURCES)
].copy()
priority = small_targets.groupby("image_path").agg(
    small_count=("class_id", "size"),
    tiny_count=("size_group", lambda values: int(sum(value == "tiny" for value in values))),
).reset_index()

extra_pool = []
for row in priority.itertuples(index=False):
    repeats = min(2, 1 + int(row.tiny_count > 0))
    extra_pool.extend([row.image_path] * repeats)
rng.shuffle(extra_pool)
extra_paths = extra_pool[:MAX_EXTRA_SAMPLES] if EXPERIMENT in {"A1_sampling", "A2_sampling_crop", "A3_sampling_crop_multiscale"} else []

def read_yolo_labels(label_path):
    labels = []
    for line in Path(label_path).read_text(encoding="utf-8", errors="ignore").splitlines():
        if line.strip():
            class_id, x, y, width, height = map(float, line.split())
            labels.append((int(class_id), x, y, width, height))
    return labels

def crop_record(image_path, target_index, output_images, output_labels):
    image_path = Path(image_path)
    labels = read_yolo_labels(label_path_for(image_path))
    with Image.open(image_path) as source_image:
        image = source_image.convert("RGB")
        image_width, image_height = image.size
        class_id, x, y, width, height = labels[target_index]
        target_width, target_height = width * image_width, height * image_height
        side = int(max(MIN_CROP_SIDE, max(target_width, target_height) * CROP_CONTEXT))
        side = min(side, image_width, image_height)
        center_x, center_y = x * image_width, y * image_height
        left = int(round(center_x - side / 2))
        top = int(round(center_y - side / 2))
        left = max(0, min(left, image_width - side))
        top = max(0, min(top, image_height - side))
        right, bottom = left + side, top + side
        cropped = image.crop((left, top, right, bottom)).resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.LANCZOS)

        converted = []
        for label_class, box_x, box_y, box_w, box_h in labels:
            x1 = (box_x - box_w / 2) * image_width
            y1 = (box_y - box_h / 2) * image_height
            x2 = (box_x + box_w / 2) * image_width
            y2 = (box_y + box_h / 2) * image_height
            clipped_x1, clipped_y1 = max(x1, left), max(y1, top)
            clipped_x2, clipped_y2 = min(x2, right), min(y2, bottom)
            intersection = max(0.0, clipped_x2 - clipped_x1) * max(0.0, clipped_y2 - clipped_y1)
            original_area = max(1e-9, (x2 - x1) * (y2 - y1))
            if intersection / original_area < MIN_BOX_VISIBILITY:
                continue
            new_x1, new_y1 = clipped_x1 - left, clipped_y1 - top
            new_x2, new_y2 = clipped_x2 - left, clipped_y2 - top
            new_x = ((new_x1 + new_x2) / 2) / side
            new_y = ((new_y1 + new_y2) / 2) / side
            new_w, new_h = (new_x2 - new_x1) / side, (new_y2 - new_y1) / side
            if new_w * IMAGE_SIZE >= 2 and new_h * IMAGE_SIZE >= 2:
                converted.append((label_class, new_x, new_y, new_w, new_h))

        if not converted:
            return None
        output_name = f"crop__{image_path.stem}__{target_index}.jpg"
        output_image = output_images / output_name
        output_label = output_labels / f"{Path(output_name).stem}.txt"
        cropped.save(output_image, quality=95, subsampling=0)
        output_label.write_text("\n".join(
            f"{label_class} {new_x:.6f} {new_y:.6f} {new_w:.6f} {new_h:.6f}"
            for label_class, new_x, new_y, new_w, new_h in converted
        ) + "\n", encoding="utf-8")
        return str(output_image.resolve())

crop_paths = []
if EXPERIMENT in {"A2_sampling_crop", "A3_sampling_crop_multiscale"}:
    crop_images = DERIVED_ROOT / EXPERIMENT / "images" / "train"
    crop_labels = DERIVED_ROOT / EXPERIMENT / "labels" / "train"
    crop_images.mkdir(parents=True, exist_ok=True)
    crop_labels.mkdir(parents=True, exist_ok=True)
    candidates = small_targets.sort_values(["area_at_640", "image_path"]).drop_duplicates("image_path")
    if len(candidates) > MAX_CROPS:
        candidates = candidates.sample(MAX_CROPS, random_state=SEED)
    for row in candidates.itertuples(index=False):
        labels = read_yolo_labels(row.label_path)
        matching = [
            index for index, (class_id, _x, _y, width, height) in enumerate(labels)
            if class_id in TARGET_CLASS_IDS and width * height * IMAGE_SIZE * IMAGE_SIZE < SMALL_AREA
        ]
        if matching:
            result = crop_record(row.image_path, matching[0], crop_images, crop_labels)
            if result:
                crop_paths.append(result)

base_paths = train_images["image_path"].tolist()
train_paths = base_paths + extra_paths + crop_paths
TRAIN_MANIFEST = CONFIG_DIR / f"train_{EXPERIMENT}.txt"
TRAIN_MANIFEST.write_text("\n".join(train_paths) + "\n", encoding="utf-8")
manifest_summary = {
    "experiment": EXPERIMENT, "original_images": len(base_paths),
    "extra_small_object_samples": len(extra_paths), "object_centered_crops": len(crop_paths),
    "effective_training_entries": len(train_paths), "seed": SEED,
}
(RESULTS_DIR / "sampling_summary.json").write_text(json.dumps(manifest_summary, indent=2), encoding="utf-8")
display(pd.DataFrame([manifest_summary]))

## 5. Train the DetectionHead candidate

All settings match the completed DetectionHead experiment except for the selected sampling/crop protocol and A3 multi-scale training. Each A1/A2/A3 experiment must start from the same YOLO26n pretrained weights for a clean ablation. `RESUME_WEIGHTS` is only for continuing the same interrupted experiment.

In [ ]:
DATA_YAML = CONFIG_DIR / f"dataset_{EXPERIMENT}.yaml"
data_config = {
    "path": str(DATASET_ROOT.resolve()),
    "train": str(TRAIN_MANIFEST.resolve()),
    "val": str((DATASET_ROOT / "images" / VAL_FOLDER).resolve()),
    "test": str((DATASET_ROOT / "images" / "test").resolve()),
    "nc": 4, "names": CLASS_NAMES,
}
DATA_YAML.write_text(yaml.safe_dump(data_config, sort_keys=False), encoding="utf-8")

RUN_NAME = f"DetectionHead_{EXPERIMENT}_seed{SEED}"
RUN_DIR = RUNS_DIR / RUN_NAME
BEST_WEIGHTS = RUN_DIR / "weights" / "best.pt"
LAST_WEIGHTS = RUN_DIR / "weights" / "last.pt"
RESULTS_CSV = RUN_DIR / "results.csv"
MULTI_SCALE = 0.25 if EXPERIMENT == "A3_sampling_crop_multiscale" else 0.0

def save_recovery(trainer):
    RECOVERY_DIR.mkdir(parents=True, exist_ok=True)
    for source in [Path(trainer.last), Path(trainer.best), Path(trainer.csv), Path(trainer.save_dir) / "args.yaml"]:
        if source.exists():
            shutil.copy2(source, RECOVERY_DIR / source.name)
    state = {
        "experiment": EXPERIMENT, "completed_epochs": int(trainer.epoch) + 1,
        "updated_utc": datetime.now(timezone.utc).isoformat(),
    }
    (RECOVERY_DIR / "state.json").write_text(json.dumps(state, indent=2), encoding="utf-8")

train_args = {
    "data": str(DATA_YAML), "epochs": EPOCHS, "imgsz": IMAGE_SIZE, "batch": BATCH,
    "workers": WORKERS, "device": TRAIN_DEVICE, "seed": SEED, "deterministic": True,
    "optimizer": "AdamW", "lr0": 0.001, "lrf": 0.01, "cos_lr": True,
    "warmup_epochs": 3, "weight_decay": 0.0005, "hsv_h": 0.015,
    "hsv_s": 0.55, "hsv_v": 0.35, "degrees": 5, "translate": 0.10,
    "scale": 0.45, "fliplr": 0.5, "mosaic": 0.5, "mixup": 0.05,
    "multi_scale": MULTI_SCALE, "close_mosaic": 10, "amp": True, "nbs": 64,
    "project": str(RUNS_DIR), "name": RUN_NAME, "exist_ok": True,
    "save": True, "save_period": SAVE_PERIOD, "val": True, "patience": 35, "plots": True,
}
manifest = {
    "experiment": EXPERIMENT, "project_name": "DetectionHead",
    "technical_architecture": "yolo26n-p2.yaml", "pretrained_transfer": "yolo26n.pt",
    "classes": CLASS_NAMES, "dataset": str(DATASET_ROOT), "training": train_args,
    "sampling": manifest_summary,
    "environment": {"python": platform.python_version(), "torch": torch.__version__, "ultralytics": ultralytics.__version__},
}
(RESULTS_DIR / "experiment_manifest.json").write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")

if RUN_TRAINING:
    if RESUME_WEIGHTS:
        resume_path = Path(RESUME_WEIGHTS).expanduser()
        if not resume_path.exists():
            raise FileNotFoundError(f"Resume checkpoint not found: {resume_path}")
        model = YOLO(str(resume_path))
        model.add_callback("on_model_save", save_recovery)
        model.train(resume=True, data=str(DATA_YAML), device=TRAIN_DEVICE, workers=WORKERS, batch=BATCH, save_period=SAVE_PERIOD, plots=True)
    elif BEST_WEIGHTS.exists() and RESULTS_CSV.exists() and len(pd.read_csv(RESULTS_CSV)) >= EPOCHS:
        print("Training is already complete.")
    else:
        model = YOLO("yolo26n-p2.yaml")
        model.load("yolo26n.pt")
        model.add_callback("on_model_save", save_recovery)
        model.train(**train_args)
else:
    print("Training disabled.")

## 6. Evaluate clean accuracy, AP by object size, and latency

The completed DetectionHead result is included as A0. A1/A2/A3 is accepted only when the small-object gain is not purchased by a material loss in overall accuracy. The same architecture means any latency difference should be measurement noise.

In [ ]:
ACTIVE_BEST = BEST_WEIGHTS if BEST_WEIGHTS.exists() else RECOVERY_DIR / "best.pt"
if not ACTIVE_BEST.exists():
    raise FileNotFoundError("The experiment best.pt was not found.")

def clean_metrics(weights, name):
    metrics = YOLO(str(weights)).val(data=str(DATA_YAML), split="val", imgsz=IMAGE_SIZE, batch=1, device=EVAL_DEVICE, plots=True)
    box = metrics.box
    record = {
        "experiment": name, "map50_95": float(box.map), "map50": float(box.map50),
        "precision_macro": float(box.mp), "recall_macro": float(box.mr),
    }
    class_ids = [int(value) for value in np.asarray(box.ap_class_index).tolist()]
    for index, class_id in enumerate(class_ids):
        record[f"{CLASS_NAMES[class_id]}_ap50_95"] = float(np.asarray(box.ap)[index])
        record[f"{CLASS_NAMES[class_id]}_recall"] = float(np.asarray(box.r)[index])
    return record

validation_df = images_df[images_df["split"] == "val"].copy()
IMAGE_TO_LABEL = dict(zip(validation_df["image_path"], validation_df["label_path"]))

def target_for_image(image_path):
    image_path = str(Path(image_path).resolve())
    with Image.open(image_path) as image:
        image_width, image_height = image.size
    boxes, labels = [], []
    for class_id, x, y, width, height in read_yolo_labels(IMAGE_TO_LABEL[image_path]):
        boxes.append([
            (x - width / 2) * image_width, (y - height / 2) * image_height,
            (x + width / 2) * image_width, (y + height / 2) * image_height,
        ])
        labels.append(class_id)
    return {
        "boxes": torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4),
        "labels": torch.tensor(labels, dtype=torch.int64),
    }

def size_metrics(weights, name):
    selected = validation_df[validation_df["source"].isin(COMPLETE_SOURCES)]
    if selected.empty:
        selected = validation_df
    model = YOLO(str(weights))
    metric = MeanAveragePrecision(box_format="xyxy", iou_type="bbox")
    paths = selected["image_path"].tolist()
    for start in range(0, len(paths), 16):
        batch = paths[start:start + 16]
        predictions = model.predict(batch, imgsz=IMAGE_SIZE, conf=0.001, iou=0.7, device=EVAL_DEVICE, verbose=False)
        predicted = [{
            "boxes": result.boxes.xyxy.detach().cpu().float(),
            "scores": result.boxes.conf.detach().cpu().float(),
            "labels": result.boxes.cls.detach().cpu().long(),
        } for result in predictions]
        metric.update(predicted, [target_for_image(path) for path in batch])
    values = metric.compute()
    return {
        "experiment": name, "ap_small": float(values["map_small"]),
        "ap_medium": float(values["map_medium"]), "ap_large": float(values["map_large"]),
    }

def profile_model(weights, name, profile_images=100):
    sample = validation_df.sample(min(profile_images, len(validation_df)), random_state=SEED)
    paths = sample["image_path"].tolist()
    model = YOLO(str(weights))
    for path in paths[:10]:
        model.predict(path, imgsz=IMAGE_SIZE, device=EVAL_DEVICE, verbose=False)
    elapsed = []
    for path in paths:
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        start = time.perf_counter()
        model.predict(path, imgsz=IMAGE_SIZE, device=EVAL_DEVICE, verbose=False)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        elapsed.append((time.perf_counter() - start) * 1000)
    p50 = float(np.percentile(elapsed, 50))
    return {
        "experiment": name, "parameters": sum(parameter.numel() for parameter in model.model.parameters()),
        "weight_size_mb": Path(weights).stat().st_size / 1024 ** 2,
        "latency_p50_ms": p50, "latency_p95_ms": float(np.percentile(elapsed, 95)),
        "fps_from_p50": 1000 / max(p50, 1e-9),
    }

candidate = clean_metrics(ACTIVE_BEST, EXPERIMENT)
candidate.update(size_metrics(ACTIVE_BEST, EXPERIMENT))
candidate.update(profile_model(ACTIVE_BEST, EXPERIMENT))
REFERENCE_A0 = {
    "experiment": "A0_current_DetectionHead",
    "map50_95": 0.4885182560, "map50": 0.8014595857,
    "precision_macro": 0.7506805829, "recall_macro": 0.7600263525,
    "helmet_ap50_95": 0.4373361055, "helmet_recall": 0.7173278880,
    "no-helmet_ap50_95": 0.4063203724, "no-helmet_recall": 0.7674613522,
    "vest_ap50_95": 0.5853005854, "vest_recall": 0.8528138528,
    "no-vest_ap50_95": 0.5251159607, "no-vest_recall": 0.7025023170,
    "ap_small": 0.2491000593, "ap_medium": 0.4426956177, "ap_large": 0.5699217319,
    "parameters": 2613152, "weight_size_mb": 5.5932,
    "latency_p50_ms": 22.9489, "latency_p95_ms": 45.5858, "fps_from_p50": 43.5750,
}
comparison = pd.DataFrame([REFERENCE_A0, candidate])
comparison["ap_small_gain_vs_A0"] = comparison["ap_small"] - REFERENCE_A0["ap_small"]
comparison["map50_95_change_vs_A0"] = comparison["map50_95"] - REFERENCE_A0["map50_95"]
comparison["latency_ratio_vs_A0"] = comparison["latency_p50_ms"] / REFERENCE_A0["latency_p50_ms"]
comparison.to_csv(RESULTS_DIR / "detectionhead_small_object_comparison.csv", index=False)
display(comparison)

keep = (
    candidate["ap_small"] - REFERENCE_A0["ap_small"] >= 0.02
    and candidate["map50_95"] - REFERENCE_A0["map50_95"] >= -0.005
    and candidate["parameters"] == REFERENCE_A0["parameters"]
)
decision = {
    "experiment": EXPERIMENT, "decision": "KEEP" if keep else "REVIEW_OR_DROP",
    "ap_small_gain": candidate["ap_small"] - REFERENCE_A0["ap_small"],
    "map50_95_change": candidate["map50_95"] - REFERENCE_A0["map50_95"],
    "architecture_unchanged": candidate["parameters"] == REFERENCE_A0["parameters"],
}
(RESULTS_DIR / "decision.json").write_text(json.dumps(decision, indent=2), encoding="utf-8")
print(decision)

## 7. Export

The exported checkpoint keeps the same YOLO26n DetectionHead architecture. Sampling, cropping, and multi-scale training add no inference layers. TensorRT FP16 should be built later on the target Jetson after the experiment passes the decision gate.

In [ ]:
exported = EXPORT_DIR / f"ppe_v2_yolo26n_detectionhead_{EXPERIMENT}_best.pt"
shutil.copy2(ACTIVE_BEST, exported)
print("Best weights:", exported)
print("Metrics:", RESULTS_DIR / "detectionhead_small_object_comparison.csv")
print("Decision:", RESULTS_DIR / "decision.json")